In [17]:
import ollama
from langchain_docling.loader import DoclingLoader
model = "gemma3:1b"

def ask_llm(prompt):
    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        stream=True
    )

    for res in response:
        print(res["message"]["content"], end="", flush=True)
FILE_PATH = r"C:/Users/ADMIN/Downloads/2346.pdf"
loader = DoclingLoader(file_path=FILE_PATH)
docs = loader.load()

The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
[INFO] 2026-06-13 10:22:12,298 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-13 10:22:12,364 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-06-13 10:22:12,365 [RapidOCR] main.py:65: Using C:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-06-13 10:22:12,512 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-13 10:22:12,591 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onn

In [18]:
print("docs variable exists:", 'docs' in globals())

if 'docs' in globals():
    print("Number of documents:", len(docs))

docs variable exists: True
Number of documents: 4


In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=60
)

texts = text_splitter.split_documents(docs)

print("Number of chunks:", len(texts))

Number of chunks: 6


In [5]:
print(texts[0])
print(texts[1])


page_content='Trips and Tours' metadata={'source': 'C:/Users/ADMIN/Downloads/2346.pdf', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/1', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 1, 'bbox': {'l': 78.0, 't': 733.0697705078124, 'r': 493.7399999999999, 'b': 699.8197705078124, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 199]}]}], 'headings': ['Trips and Tours'], 'origin': {'mimetype': 'application/pdf', 'binary_hash': 14683495203959894484, 'filename': '2346.pdf'}}}
page_content='Trips and tours provide opportunities to explore new places, experience different cultures, and create lasting memories. They can be organized for leisure, education, adventure, or business purposes.' metadata={'source': 'C:/Users/ADMIN/Downloads/2346.pdf', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_

In [20]:
#pip install sentence-transfromer
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7418.07it/s]


In [7]:
print("Embeddings Loaded Successfully")

Embeddings Loaded Successfully


In [21]:
from langchain_chroma import Chroma
vector_store = Chroma(
    collection_name = "sample_RAG",
    embedding_function = embeddings,
    persist_directory= "./chroma_langchain_db",
)

In [23]:
vector_store._collection.count()

0

In [24]:
from langchain_community.vectorstores.utils import filter_complex_metadata
texts = filter_complex_metadata(texts)

In [25]:
texts[5]


Document(metadata={'source': 'C:/Users/ADMIN/Downloads/2346.pdf'}, page_content='Conclusion\nTrips and tours enrich life by offering new experiences, knowledge, and enjoyment. Proper planning helps travelers make the most of their journeys.')

In [26]:
vector_store.add_documents(texts)

['8c270868-3b71-4eb4-ac0d-2e9032ae0d72',
 '7bfe5e77-9e14-4016-8735-f5e44d7438ce',
 'b969a0bb-7b1b-483a-9f6a-45c63fb882a5',
 '49c68350-93ba-4711-9c14-90f9159161c9',
 '79b6987f-2cb5-49df-8c58-3e5b44a583c9',
 '27115bbc-95d2-49fc-878f-f135ec06256f']

In [27]:
vector_store._collection.count()

6

In [28]:
vector_store._collection.get(ids=["8c270868-3b71-4eb4-ac0d-2e9032ae0d72"], include=["embeddings","documents"])

{'ids': ['8c270868-3b71-4eb4-ac0d-2e9032ae0d72'],
 'embeddings': array([[ 3.05540524e-02, -2.95282956e-02,  1.02018014e-01,
         -4.82255742e-02, -6.69034570e-02,  4.80890125e-02,
          2.26267781e-02,  2.40097847e-02, -4.87389863e-02,
         -4.74236198e-02, -7.89012015e-03, -3.16834040e-02,
          9.30941664e-03,  4.07480188e-02, -1.06189419e-02,
          1.63500998e-02, -1.33313239e-02, -4.13984917e-02,
         -3.67435589e-02,  4.08947887e-03,  2.49087866e-02,
          1.86230112e-02,  1.15140714e-02,  1.48683740e-02,
          4.04150784e-03,  5.94755374e-02,  2.92341672e-02,
         -4.40089381e-04, -5.26343845e-02, -7.45315254e-02,
         -3.52112669e-03, -4.70733494e-02, -4.06354554e-02,
         -4.67066746e-03, -2.87800170e-02,  3.96339074e-02,
         -3.05758882e-03,  2.78560184e-02, -3.74660641e-02,
          4.24471637e-03,  4.84351739e-02,  3.38109629e-03,
          3.73578630e-02, -2.08997484e-02,  3.40130902e-03,
          1.19088718e-03, -6.0580102

In [29]:
retriever = vector_store.as_retriever()

In [30]:
query = "What is this document about?"

retrieved_docs = retriever.invoke(query)

context = "\n\n".join(
    [doc.page_content for doc in retrieved_docs]
)

print(context)

- Improving confidence and independence
- Enjoying nature, history, and local attractions

Trips and tours provide opportunities to explore new places, experience different cultures, and create lasting memories. They can be organized for leisure, education, adventure, or business purposes.

Trips and Tours

Conclusion
Trips and tours enrich life by offering new experiences, knowledge, and enjoyment. Proper planning helps travelers make the most of their journeys.


In [32]:
prompt = f"""
Answer the question using only the provided context.

Context:
{context}

Question:
{query}

Answer:
"""

In [33]:
print(query)
print(context)
ask_llm(prompt)

What is this document about?
- Improving confidence and independence
- Enjoying nature, history, and local attractions

Trips and tours provide opportunities to explore new places, experience different cultures, and create lasting memories. They can be organized for leisure, education, adventure, or business purposes.

Trips and Tours

Conclusion
Trips and tours enrich life by offering new experiences, knowledge, and enjoyment. Proper planning helps travelers make the most of their journeys.
This document focuses on promoting and encouraging travel—particularly through trips and tours – because they benefit from enriching individuals' lives with new experiences, opportunities to learn, and enjoy nature and local attractions.  Essentially it highlights the positive aspects of travel.
